# Question to Cypher — Evaluation

Notebook untuk mengevaluasi NL-to-Cypher pipeline:
1. Load prompt template dari Google Sheets
2. Load test data dari Google Sheets / CSV
3. Untuk setiap pertanyaan: generate Cypher via LLM (langsung, tanpa backend)
4. Run actual Cypher di Neo4j → `ACTUAL_QUERY_RESULT`
5. Bandingkan dengan `EXPECTED_QUERY_RESULT` dari test data
6. Simpan hasil ke sheet experiment baru

In [6]:
# === Setup ===
import sys
import json
import os
import re
import time
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd

PROJECT_ROOT = Path(os.getcwd()).parent.parent if 'notebooks' in str(Path(os.getcwd())) else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
load_dotenv()

print(f'Project root: {PROJECT_ROOT}')

Project root: d:\TA\llm-driven-legal-kg-visualization


## Step 0: Configuration

In [7]:
# === Configuration ===
EXPERIMENT_ID = "Q2C_002"      # Unique ID per eksperimen
PROMPT_ID = "PROMPT_4"   # ID prompt di sheet QUESTION_TO_CYPHER_PROMPT

# Sumber test data
TEST_DATA_SOURCE = "gsheets"   # 'csv' atau 'gsheets'
CSV_PATH = r"d:\TA\[SFT] Regulation Fine Tuning for Knowledge Graph - QUESTION_TO_CYPHER_QUERY_DATA_TEST.csv"
GSHEETS_TEST_SHEET = "QUESTION_TO_CYPHER_QUERY_DATA_TEST"

# Output
WRITE_TO_GSHEETS = True
EXPERIMENT_SHEET_NAME = f"EXP_{EXPERIMENT_ID}"

print(f'Experiment: {EXPERIMENT_ID}')
print(f'Prompt: {PROMPT_ID}')
print(f'Test data: {TEST_DATA_SOURCE}')
print(f'Output sheet: {EXPERIMENT_SHEET_NAME}')

Experiment: Q2C_002
Prompt: PROMPT_4
Test data: gsheets
Output sheet: EXP_Q2C_002


## Step 1: Load Prompt Template from Google Sheets

In [8]:
# === Load Prompt from GSheets ===
from modules.prompt_fetcher import fetch_question_to_cypher_prompt

prompt_data = fetch_question_to_cypher_prompt(PROMPT_ID)

SYSTEM_PROMPT = prompt_data['SYSTEM_PROMPT']
USER_PROMPT_TEMPLATE = prompt_data.get('USER_PROMPT', 'Question: {question}')

print(f'System prompt length: {len(SYSTEM_PROMPT)} chars')
print(f'User prompt template length: {len(USER_PROMPT_TEMPLATE)} chars')
print(f'\n--- System Prompt Preview ---')
print(SYSTEM_PROMPT)
print('...')

2026-04-24 17:44:02,009 - INFO - Fetching prompt 'PROMPT_4' from sheet 'QUESTION_TO_CYPHER_QUERY_PROMPT_TEMPLATE'...
2026-04-24 17:44:02,009 - INFO - Retrieving worksheet 'QUESTION_TO_CYPHER_QUERY_PROMPT_TEMPLATE' from spreadsheet ID '1oN5kMN_OI8WyITAQgJ3-S_0GlzraXug8p2tMKSmq7u0'...
2026-04-24 17:44:03,937 - INFO - Successfully loaded 4 rows from worksheet 'QUESTION_TO_CYPHER_QUERY_PROMPT_TEMPLATE'.
2026-04-24 17:44:03,940 - INFO - Loaded prompt 'PROMPT_4': ['PROMPT_ID', 'SYSTEM_PROMPT', 'USER_PROMPT', 'NOTES']


System prompt length: 5858 chars
User prompt template length: 125 chars

--- System Prompt Preview ---
You are a Cypher query generator for an Indonesian legal Knowledge Graph in Neo4j.
Given a user question about Indonesian law, generate a Cypher query to retrieve the relevant data.

{KG_SCHEMA}

## STRICT OUTPUT RULES
1. Output ONLY the raw Cypher query. NO markdown, NO ```, NO explanation.
2. For label filters: use toLower() with exact match (=) for Bab labels (e.g., toLower(b.label) = 'bab vii') and single-number Pasal labels (e.g., toLower(p.label) = 'pasal 3'). Use CONTAINS only for multi-word/ayat searches (e.g., p.label CONTAINS 'Pasal 27').
3. ALWAYS end with LIMIT 25.
4. Query must be syntactically valid (balanced parentheses, MATCH + RETURN).
5. Use toLower() for case-insensitive matching.
6. Node labels do NOT contain regulation names like "UU ITE" or "UU No. 11 Tahun 2008". Strip these from the search filter. Example: "Pasal 1 UU ITE" → filter by 'pasal 1' only.

## QUERY 

## Step 2: Load Test Data

In [9]:
# === Load Test Data ===
if TEST_DATA_SOURCE == "gsheets":
    from modules.google_sheets_utils import GoogleUtil
    gu = GoogleUtil(
        private_key=os.getenv('GOOGLE_SHEETS_PRIVATE_KEY', ''),
        client_email=os.getenv('GOOGLE_SHEETS_CLIENT_EMAIL', '')
    )
    spreadsheet_id = os.getenv('GOOGLE_SPREADSHEET_ID', '')
    test_df = gu.load_dataframe_from_sheet(spreadsheet_id, GSHEETS_TEST_SHEET)
else:
    test_df = pd.read_csv(CSV_PATH)

print(f'Loaded {len(test_df)} test cases')
print(f'Columns: {list(test_df.columns)}')
print(f'Categories: {test_df["CATEGORY"].value_counts().to_dict()}')
test_df[['TEST_ID', 'QUESTION', 'CATEGORY']].head(10)

2026-04-24 17:44:03,953 - INFO - Retrieving worksheet 'QUESTION_TO_CYPHER_QUERY_DATA_TEST' from spreadsheet ID '1oN5kMN_OI8WyITAQgJ3-S_0GlzraXug8p2tMKSmq7u0'...
2026-04-24 17:44:05,496 - INFO - Successfully loaded 42 rows from worksheet 'QUESTION_TO_CYPHER_QUERY_DATA_TEST'.


Loaded 42 test cases
Columns: ['TEST_ID', 'QUESTION', 'CATEGORY', 'EXPECTED_CYPHER_QUERY', 'EXPECTED_QUERY_RESULT', 'FORMATTED_EXPECTED_QUERY_RESULT']
Categories: {'hierarki': 14, 'mengatur': 7, 'sanksi': 5, 'definisi': 5, 'entitas': 4, 'relasi': 4, 'keyword': 3}


,TEST_ID,QUESTION,CATEGORY
0,Q2C_001,Apa yang diatur Pasal 27?,mengatur
1,Q2C_002,Apa yang diatur dalam Pasal 30?,mengatur
2,Q2C_003,Pasal 28 mengatur tentang apa?,mengatur
3,Q2C_004,Perbuatan apa yang diatur Pasal 32?,mengatur
4,Q2C_005,Apa yang dilarang dalam Pasal 35?,mengatur
5,Q2C_006,Apa sanksi pencemaran nama baik?,sanksi
6,Q2C_007,Apa sanksi pelanggaran Pasal 27?,sanksi
7,Q2C_008,Berapa denda untuk pelanggaran Pasal 30?,sanksi
8,Q2C_009,Apa hukuman untuk pelanggaran Pasal 31?,sanksi
9,Q2C_010,Sanksi apa untuk pelanggaran Pasal 32?,sanksi


## Step 3: Connect to Neo4j & Setup LLM

In [10]:
# === Connect to Neo4j ===
from neo4j import GraphDatabase

NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', '')
NEO4J_DATABASE = os.getenv('NEO4J_DATABASE', 'neo4j')

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
with driver.session(database=NEO4J_DATABASE) as session:
    session.run('RETURN 1 AS test')
print(f'Connected to Neo4j ({NEO4J_DATABASE})')

# === Setup Gemini LLM (langsung, tanpa backend) ===
import google.generativeai as genai

genai.configure(api_key=os.getenv('GEMINI_API_KEY', ''))
model = genai.GenerativeModel('gemini-2.5-flash')
print('LLM model ready: gemini-2.5-flash')

Connected to Neo4j (experiment)
LLM model ready: gemini-2.5-flash


c:\Users\daffarafi\miniconda3\envs\ta-skripsi\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\daffarafi\AppData\Local\Temp\ipykernel_3624\1618597011.py:15: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


## Step 4: Define Helper Functions

In [11]:
def clean_cypher(text: str) -> str:
    """Strip markdown code blocks from LLM output."""
    pattern = r'```(?:cypher|sql|plaintext)?\s*\n?(.*?)```'
    match = re.search(pattern, text, re.DOTALL)
    if match:
        return match.group(1).strip()
    if text.startswith('```'):
        lines = text.split('\n')
        return '\n'.join(lines[1:]).strip().rstrip('`')
    return text.strip()


def is_valid_cypher(query: str) -> bool:
    """Basic Cypher validation."""
    if not query or 'RETURN' not in query.upper():
        return False
    if query.count('(') != query.count(')'):
        return False
    if query.count('[') != query.count(']'):
        return False
    return True


def generate_cypher(question: str, max_retries: int = 2) -> str:
    """Generate Cypher query from question using LLM."""
    user_prompt = USER_PROMPT_TEMPLATE.replace('{question}', question)
    
    for attempt in range(max_retries):
        try:
            response = model.generate_content(
                [SYSTEM_PROMPT, user_prompt],
                generation_config={'temperature': 0.0, 'max_output_tokens': 2048},
            )
            cypher = clean_cypher(response.text.strip())
            if is_valid_cypher(cypher):
                return cypher
        except Exception as e:
            if 'quota' in str(e).lower() or '429' in str(e):
                wait = 5 * (attempt + 1)
                print(f' [rate limit, waiting {wait}s]', end='')
                time.sleep(wait)
            else:
                raise
    return cypher  # return last attempt even if invalid


def run_cypher(query: str) -> list:
    """Run Cypher query and return results as list of dicts."""
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(query)
        return [dict(record) for record in result]


def truncate_content(results: list, max_len: int = 200) -> list:
    """Truncate long text fields for readability."""
    long_fields = ('content', 'isi', 'detail', 'definisi')
    truncated = []
    for row in results:
        new_row = {}
        for k, v in row.items():
            if isinstance(v, str) and k in long_fields and len(v) > max_len:
                new_row[k] = v[:max_len] + '...'
            else:
                new_row[k] = v
        truncated.append(new_row)
    return truncated


def compare_results(expected_json: str, actual: list) -> dict:
    """Compare expected vs actual query results.
    
    Returns dict with: match (bool), score (float), notes (str)
    """
    try:
        expected = json.loads(expected_json) if isinstance(expected_json, str) else expected_json
    except (json.JSONDecodeError, TypeError):
        return {'match': False, 'score': 0.0, 'notes': 'Failed to parse expected JSON'}
    
    if not expected and not actual:
        return {'match': True, 'score': 1.0, 'notes': 'Both empty'}
    if not expected or not actual:
        return {'match': False, 'score': 0.0, 'notes': f'expected={len(expected) if expected else 0}, actual={len(actual)}'}
    
    # Normalize: convert all values to strings for comparison
    def normalize(rows):
        normalized = []
        for row in rows:
            n = {}
            for k, v in row.items():
                if isinstance(v, list):
                    n[k] = str(v)
                elif isinstance(v, str):
                    # Truncate for comparison (expected may be truncated)
                    n[k] = v[:200] if len(v) > 200 else v
                else:
                    n[k] = str(v) if v is not None else ''
            normalized.append(n)
        return normalized
    
    norm_expected = normalize(expected)
    norm_actual = normalize(actual)
    
    # Check key overlap: do expected keys match actual keys?
    exp_keys = set(norm_expected[0].keys()) if norm_expected else set()
    act_keys = set(norm_actual[0].keys()) if norm_actual else set()
    key_match = exp_keys == act_keys
    
    # Count how many expected rows are found in actual
    common_keys = exp_keys & act_keys
    if not common_keys:
        return {'match': False, 'score': 0.0, 'notes': f'No common keys: expected={exp_keys}, actual={act_keys}'}
    
    found = 0
    for exp_row in norm_expected:
        for act_row in norm_actual:
            if all(exp_row.get(k, '') == act_row.get(k, '') for k in common_keys):
                found += 1
                break
    
    recall = found / len(norm_expected)
    precision = found / len(norm_actual) if norm_actual else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    match = (recall == 1.0 and precision == 1.0)
    notes = f'found={found}/{len(norm_expected)} recall={recall:.1%} precision={precision:.1%} F1={f1:.1%}'
    if not key_match:
        notes += f' [key mismatch: exp={exp_keys}, act={act_keys}]'
    
    return {'match': match, 'score': f1, 'notes': notes}


print('Helper functions defined')

Helper functions defined


## Step 5: Run Evaluation

In [12]:
# === Run All Test Cases ===
results = []

for idx, row in test_df.iterrows():
    test_id = row['TEST_ID']
    question = row['QUESTION']
    category = row['CATEGORY']
    expected_cypher = row.get('EXPECTED_CYPHER_QUERY', '')
    expected_result_json = row.get('EXPECTED_QUERY_RESULT', '[]')
    
    print(f'[{idx+1}/{len(test_df)}] {test_id}: {question[:60]}...', end='')
    
    # --- Generate actual Cypher via LLM ---
    try:
        actual_cypher = generate_cypher(question)
    except Exception as e:
        actual_cypher = ''
        print(f' LLM_ERROR: {e}')
    
    # --- Run actual Cypher against Neo4j ---
    actual_result = []
    cypher_error = ''
    try:
        if actual_cypher and is_valid_cypher(actual_cypher):
            actual_result = run_cypher(actual_cypher)
            actual_result = truncate_content(actual_result)
        else:
            cypher_error = 'Invalid Cypher generated'
    except Exception as e:
        cypher_error = str(e)[:200]
    
    # --- Compare expected vs actual ---
    comparison = compare_results(expected_result_json, actual_result)
    
    status = 'PASS' if comparison['match'] else ('ERROR' if cypher_error else 'FAIL')
    icon = {'PASS': '\u2705', 'FAIL': '\u274c', 'ERROR': '\u26a0\ufe0f'}[status]
    print(f' {icon} {status} - {comparison["notes"]}')
    
    # --- Format results ---
    try:
        exp_obj = json.loads(expected_result_json) if isinstance(expected_result_json, str) else expected_result_json
        formatted_expected = json.dumps(exp_obj, indent=2, ensure_ascii=False)
    except:
        formatted_expected = str(expected_result_json)
    
    formatted_actual = json.dumps(actual_result, indent=2, ensure_ascii=False)
    
    results.append({
        'TEST_ID': test_id,
        'QUESTION': question,
        'CATEGORY': category,
        'EXPECTED_CYPHER_QUERY': expected_cypher,
        'EXPECTED_QUERY_RESULT': expected_result_json,
        'FORMATTED_EXPECTED_QUERY_RESULT': formatted_expected,
        'ACTUAL_CYPHER_QUERY': actual_cypher,
        'ACTUAL_QUERY_RESULT': json.dumps(actual_result, ensure_ascii=False),
        'FORMATTED_ACTUAL_QUERY_RESULT': formatted_actual,
        'RESULT': status,
        'SCORE': round(comparison['score'], 4),
        'NOTES': (cypher_error + ' | ' if cypher_error else '') + comparison['notes'],
    })
    
    # Rate limit protection
    time.sleep(1)

results_df = pd.DataFrame(results)
print(f'\n=== Done: {len(results)} test cases evaluated ===')

[1/42] Q2C_001: Apa yang diatur Pasal 27?... ✅ PASS - found=4/4 recall=100.0% precision=100.0% F1=100.0%
[2/42] Q2C_002: Apa yang diatur dalam Pasal 30?... ✅ PASS - found=3/3 recall=100.0% precision=100.0% F1=100.0%
[3/42] Q2C_003: Pasal 28 mengatur tentang apa?... ✅ PASS - found=2/2 recall=100.0% precision=100.0% F1=100.0%
[4/42] Q2C_004: Perbuatan apa yang diatur Pasal 32?... ❌ FAIL - expected=3, actual=0
[5/42] Q2C_005: Apa yang dilarang dalam Pasal 35?... ✅ PASS - found=1/1 recall=100.0% precision=100.0% F1=100.0%
[6/42] Q2C_006: Apa sanksi pencemaran nama baik?... ✅ PASS - found=6/6 recall=100.0% precision=100.0% F1=100.0%
[7/42] Q2C_007: Apa sanksi pelanggaran Pasal 27?... ✅ PASS - found=6/6 recall=100.0% precision=100.0% F1=100.0%
[8/42] Q2C_008: Berapa denda untuk pelanggaran Pasal 30?... ✅ PASS - found=6/6 recall=100.0% precision=100.0% F1=100.0%
[9/42] Q2C_009: Apa hukuman untuk pelanggaran Pasal 31?... ✅ PASS - found=5/5 recall=100.0% precision=100.0% F1=100.0%
[10/42] Q2C_0

## Step 6: Summary

In [13]:
# === Overall Summary ===
total = len(results_df)
passed = len(results_df[results_df['RESULT'] == 'PASS'])
failed = len(results_df[results_df['RESULT'] == 'FAIL'])
errors = len(results_df[results_df['RESULT'] == 'ERROR'])
avg_score = results_df['SCORE'].mean()

border = '=' * 50
print(f'{border}')
print(f'  Q2C Evaluation Report')
print(f'{border}')
print(f'  Experiment: {EXPERIMENT_ID}')
print(f'  Prompt:     {PROMPT_ID}')
print(f'  Database:   {NEO4J_DATABASE}')
print(f'{border}')
print(f'  Total:    {total:>3d}')
print(f'  Pass:     {passed:>3d}  ({passed/total:.1%})')
print(f'  Fail:     {failed:>3d}  ({failed/total:.1%})')
print(f'  Error:    {errors:>3d}')
print(f'  Avg F1:   {avg_score:.2%}')
print(f'{border}')

  Q2C Evaluation Report
  Experiment: Q2C_002
  Prompt:     PROMPT_4
  Database:   experiment
  Total:     42
  Pass:      38  (90.5%)
  Fail:       4  (9.5%)
  Error:      0
  Avg F1:   94.05%


In [14]:
# === Per-Category Breakdown ===
header = f'{"CATEGORY":<20s} {"PASS":>5s} {"FAIL":>5s} {"ERR":>5s} {"TOTAL":>5s} {"RATE":>8s} {"AVG_F1":>10s}'
print(f'\n{header}')
sep = '-' * 65
print(sep)

for cat in results_df['CATEGORY'].unique():
    cat_df = results_df[results_df['CATEGORY'] == cat]
    cp = len(cat_df[cat_df['RESULT'] == 'PASS'])
    cf = len(cat_df[cat_df['RESULT'] == 'FAIL'])
    ce = len(cat_df[cat_df['RESULT'] == 'ERROR'])
    ct = len(cat_df)
    cr = cp / ct
    ca = cat_df['SCORE'].mean()
    print(f'{cat:<20s} {cp:>5d} {cf:>5d} {ce:>5d} {ct:>5d} {cr:>7.1%} {ca:>9.2%}')

print(sep)
print(f'{"TOTAL":<20s} {passed:>5d} {failed:>5d} {errors:>5d} {total:>5d} {passed/total:>7.1%} {avg_score:>9.2%}')


CATEGORY              PASS  FAIL   ERR TOTAL     RATE     AVG_F1
-----------------------------------------------------------------
mengatur                 6     1     0     7   85.7%    85.71%
sanksi                   5     0     0     5  100.0%   100.00%
hierarki                13     1     0    14   92.9%   103.57%
definisi                 5     0     0     5  100.0%   100.00%
entitas                  2     2     0     4   50.0%    50.00%
relasi                   4     0     0     4  100.0%   100.00%
keyword                  3     0     0     3  100.0%   100.00%
-----------------------------------------------------------------
TOTAL                   38     4     0    42   90.5%    94.05%


In [15]:
# === Detail: Failed/Error Test Cases ===
failed_df = results_df[results_df['RESULT'] != 'PASS']

if len(failed_df) == 0:
    print('All test cases passed!')
else:
    print(f'\nFailed/Error test cases ({len(failed_df)}):\n')
    for _, r in failed_df.iterrows():
        print(f"  {r['TEST_ID']} [{r['CATEGORY']}] - {r['QUESTION'][:60]}")
        print(f"    Result: {r['RESULT']}, F1: {r['SCORE']:.2%}")
        print(f"    Notes: {r['NOTES'][:120]}")
        print()


Failed/Error test cases (4):

  Q2C_004 [mengatur] - Perbuatan apa yang diatur Pasal 32?
    Result: FAIL, F1: 0.00%
    Notes: expected=3, actual=0

  Q2C_021 [entitas] - Pasal 45 berlaku untuk siapa?
    Result: FAIL, F1: 0.00%
    Notes: expected=3, actual=0

  Q2C_023 [entitas] - Pasal 15 berlaku untuk siapa saja?
    Result: FAIL, F1: 0.00%
    Notes: expected=2, actual=0

  Q2C_042 [hierarki] - Pasal 30 punya berapa ayat?
    Result: FAIL, F1: 150.00%
    Notes: found=3/3 recall=100.0% precision=300.0% F1=150.0% [key mismatch: exp={'isi', 'pasal', 'ayat'}, act={'pasal', 'jumlah_ay



## Step 7: Save Results

In [16]:
# === Save to CSV ===
output_dir = 'data/evaluation'
os.makedirs(output_dir, exist_ok=True)

output_csv = f'{output_dir}/q2c_eval_{EXPERIMENT_ID}.csv'
results_df.to_csv(output_csv, index=False, encoding='utf-8')
print(f'Results saved to: {output_csv}')

Results saved to: data/evaluation/q2c_eval_Q2C_002.csv


In [17]:
# === Write to Google Sheets ===
if WRITE_TO_GSHEETS:
    from modules.google_sheets_utils import GoogleUtil, GoogleSheetsWriter

    gu = GoogleUtil(
        private_key=os.getenv('GOOGLE_SHEETS_PRIVATE_KEY', ''),
        client_email=os.getenv('GOOGLE_SHEETS_CLIENT_EMAIL', '')
    )
    spreadsheet_id = os.getenv('GOOGLE_SPREADSHEET_ID', '')

    writer = GoogleSheetsWriter(
        google_util=gu,
        sheet_id=spreadsheet_id,
        worksheet_name=EXPERIMENT_SHEET_NAME,
        batch_size=5,
    )

    result = writer.write_dataframe(results_df)
    print(f'Written to Google Sheets: {EXPERIMENT_SHEET_NAME}')
    print(f'   Success: {result.successful_rows}, Failed: {result.failed_rows}')
else:
    print(f'WRITE_TO_GSHEETS = False, skipping.')
    print(f'Set WRITE_TO_GSHEETS = True to write to sheet "{EXPERIMENT_SHEET_NAME}"')

  0%|          | 0/9 [00:00<?, ?it/s]

2026-04-24 17:46:49,384 - INFO - Successfully wrote row 1/42
2026-04-24 17:46:52,051 - INFO - Successfully wrote row 2/42
2026-04-24 17:46:54,802 - INFO - Successfully wrote row 3/42
2026-04-24 17:46:57,008 - INFO - Successfully wrote row 4/42
2026-04-24 17:46:59,019 - INFO - Successfully wrote row 5/42
 11%|█         | 1/9 [00:14<01:55, 14.42s/it]2026-04-24 17:47:05,424 - INFO - Successfully wrote row 6/42
2026-04-24 17:47:09,113 - INFO - Successfully wrote row 7/42
2026-04-24 17:47:11,494 - INFO - Successfully wrote row 8/42
2026-04-24 17:47:16,613 - INFO - Successfully wrote row 9/42
2026-04-24 17:47:19,001 - INFO - Successfully wrote row 10/42
 22%|██▏       | 2/9 [00:34<02:03, 17.69s/it]2026-04-24 17:47:22,971 - INFO - Successfully wrote row 11/42
2026-04-24 17:47:25,144 - INFO - Successfully wrote row 12/42
2026-04-24 17:47:27,562 - INFO - Successfully wrote row 13/42
2026-04-24 17:47:30,436 - INFO - Successfully wrote row 14/42
2026-04-24 17:47:32,726 - INFO - Successfully wrote

Written to Google Sheets: EXP_Q2C_002
   Success: 42, Failed: 0


In [18]:
# === Cleanup ===
driver.close()
print('Neo4j connection closed.')

Neo4j connection closed.
